<a href="https://colab.research.google.com/github/miaflynn/CYPLAN255-Final-Project/blob/main/02b_processing_w_tracts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [129]:
import pandas as pd
import geopandas as gpd
import numpy as np

#added more that we use in lab
import os
%matplotlib inline
import matplotlib.pyplot as plt
from shapely.geometry import LineString


In [130]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [131]:
!ls /content/drive
!ls /content/drive/Shareddrives

MyDrive  Shareddrives


In [132]:
import pandas as pd

gdf = pd.read_csv("/content/drive/MyDrive/C255_final_project/cleaned/rbl_total_cleaned_all_dates.csv")

In [133]:
gdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 206872 entries, 0 to 206871
Data columns (total 40 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   Unnamed: 0                         206872 non-null  int64  
 1   uniqueid                           206872 non-null  object 
 2   business_account_number            206872 non-null  int64  
 3   location_id                        206872 non-null  object 
 4   ownership_name                     206872 non-null  object 
 5   dba_name                           206683 non-null  object 
 6   street_address                     206872 non-null  object 
 7   city                               206872 non-null  object 
 8   state                              206860 non-null  object 
 9   source_zipcode                     206849 non-null  float64
 10  business_start_date                206872 non-null  object 
 11  business_end_date                  8432

In [134]:
# # adding another plot and importing a shpfile of SF so we can see where the points are in SF

# import matplotlib.pyplot as plt

# # Importing SF geometry
# # URL for 2025 TIGER/Line Place boundaries - info here on
# ## how to use: https://www.census.gov/geographies/mapping-files/time-series/geo/tiger-line-file.html
url = "https://www2.census.gov/geo/tiger/TIGER2025/PLACE/tl_2025_06_place.zip"

places = gpd.read_file(url)

# Filtered to SF
sf_poly = places[
    (places["NAME"] == "San Francisco") &
    (places["STATEFP"] == "06")   # 06 = California
]


# eproject to same as our gdf
sf_poly = sf_poly.to_crs(epsg=4326)

## **Tract Level Analysis**

In [135]:
#I added a shp file with census tracts to our folder

tracts_gdf = gpd.read_file(
    "/content/drive/MyDrive/C255_final_project/cb_2020_06_tract_500k"
)

In [136]:
#checking to make sure all good

tracts_gdf.columns

# simplifying
tracts_gdf = tracts_gdf[["NAME", "NAMELSAD", "STATE_NAME","GEOID", "geometry"]]


In [137]:
# was getting error with "index_right" - so troubleshooted this
tracts_gdf = tracts_gdf.drop(columns=["index_right", ], errors="ignore")
gdf = gdf.drop(columns=["index_right"], errors="ignore")

In [138]:
# Joining the gdf and tract gdf so we can summarize within tracts

gdf = gdf.set_geometry(
    gpd.points_from_xy(gdf.lon, gdf.lat)
)

gdf = gdf.set_crs(epsg=4326)
tracts_gdf = tracts_gdf.to_crs(epsg=4326)

# joining within
business_tracts_gdf = gpd.sjoin(gdf, tracts_gdf, how="left", predicate="within")

In [139]:
business_tracts_gdf.columns.tolist()

cols_to_keep = [
    "uniqueid",
    "dba_name",
    "business_account_number",
    "naics_code",
    "naics_code_description",
    "lic_code",
    "lic_code_description",
    "neighborhoods_analysis_boundaries",
    "year",
    "status",
    "location_start_date",
    "location_end_date",
    "GEOID",
    "geometry",
    "lon", "lat"
]

business_tracts_gdf = business_tracts_gdf[cols_to_keep]

In [140]:
# I'm counting the number of businesses per tract per year, separated by closing and openings status
tract_year = (
    business_tracts_gdf
    .groupby(["GEOID", "year", "status"])
    .size()
    .reset_index(name="count")
    .pivot(index=["GEOID", "year"], columns="status", values="count")
    .fillna(0)
    .reset_index()
)

In [141]:
tracts_plot = tracts_gdf[["GEOID", "geometry"]].merge(
    tract_year,
    on="GEOID",
    how="left"
).fillna(0)

In [142]:
tracts_plot.info()
tracts_plot = gpd.clip(tracts_plot, sf_poly)

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 19285 entries, 0 to 19284
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   GEOID     19285 non-null  object  
 1   geometry  19285 non-null  geometry
 2   year      19285 non-null  float64 
 3   closed    19285 non-null  float64 
 4   opened    19285 non-null  float64 
dtypes: float64(3), geometry(1), object(1)
memory usage: 753.4+ KB


In [143]:
tracts_plot["year"] = tracts_plot["year"].astype(int)
tracts_plot = tracts_plot.sort_values(["GEOID", "year"])

In [144]:
print(tracts_plot.columns)

Index(['GEOID', 'geometry', 'year', 'closed', 'opened'], dtype='object')


In [145]:
# opened = tracts_plot[tracts_plot["status"] == "opened"]
# closed = tracts_plot[tracts_plot["status"] == "closed"]

In [146]:
# This is the code I used to make some interactive visualizations, but the file size was preventing me from uploading to Github. I commented this out for now so I could upload.

# fig_closed = px.choropleth_mapbox(
#     closed,
#     geojson=closed.set_index("GEOID").__geo_interface__,
#     locations="GEOID",
#     color="count",
#     hover_name="GEOID",
#     animation_frame="year",
#     mapbox_style="carto-positron",
#     zoom=10,
#     center={"lat": 37.7749, "lon": -122.4194},  # centering on SF becuase it wasn't when I tested
#     color_continuous_scale="Reds",
#     height=600
# )

# fig_closed.update_layout(title="Business Closings by Census Tract (2016–2025)")
# fig_closed.show()

In [147]:
# fig_opened = px.choropleth_mapbox(
#     opened,
#     geojson=closed.set_index("GEOID").__geo_interface__,
#     locations="GEOID",
#     color="count",
#     hover_name="GEOID",
#     animation_frame="year",
#     mapbox_style="carto-positron",
#     zoom=10,
#     center={"lat": 37.7749, "lon": -122.4194},  # centering on SF becuase it wasn't when I tested
#     color_continuous_scale="Blues",
#     height=600
# )

# fig_opened.update_layout(title="Business Openings by Census Tract (2016–2025)")
# fig_opened.show()

Need to baseline so we don't have viz skewed by areas with higher # of overall businesses

In [148]:
# Reformatting so each row is a unique count per tract, year, and status (so we don't have closings and openings for busness in one row)
business_tracts_separated = (
    business_tracts_gdf
    .groupby(["GEOID", "year", "status"])
    .size()
    .reset_index(name="count")
)

In [149]:
# Baselining by 2016 (but this can change - just testing it out)

baseline = business_tracts_separated[business_tracts_separated["year"] == 2016][["GEOID", "status", "count"]].rename(columns={"count": "baseline"}) # makes new df and filtering for only 2016 so we can use as baseline
business_tracts_separated = business_tracts_separated.merge(baseline, on=["GEOID", "status"], how="left") # joining this new baseline df to business tracts df so we can calculate change
business_tracts_separated["change_from_2016"] = business_tracts_separated["count"] - business_tracts_separated["baseline"] # subtracts change from baseline


In [150]:
tracts_plot = tracts_gdf[["GEOID", "geometry"]].merge(
    business_tracts_separated,
    on="GEOID",
    how="left"
).fillna(0)

In [151]:
opened = tracts_plot[tracts_plot["status"] == "opened"]
closed = tracts_plot[tracts_plot["status"] == "closed"]

In [152]:
# sanity check on changes

opened["change_from_2016"].describe()

# most tracts didn't have major changes (average of 3 fewer openings compared to 2016)
# - but some tracts had a major change (both pos and neg compared to 2016)- these extremes will skew our viz

,change_from_2016
count,10349.000000
mean,-31.169292
std,66.141243
min,-820.000000
25%,-33.000000
50%,-21.000000
75%,-10.000000
max,426.000000


In [153]:
closed["change_from_2016"].describe()

,change_from_2016
count,3898.000000
mean,4.875321
std,29.072937
min,-240.000000
25%,-3.000000
50%,2.000000
75%,13.000000
max,510.000000


In [154]:
# fig = px.choropleth_mapbox(
#     opened,
#     geojson=tracts_plot.set_index("GEOID").__geo_interface__,
#     locations="GEOID",
#     color="change_from_2016",
#     hover_name="GEOID",
#     animation_frame="year",
#     mapbox_style="carto-positron",
#     zoom=11,
#     center={"lat": 37.7749, "lon": -122.4194},
#     color_continuous_scale="Blues",
#     range_color=[-20, 20], # adding this so that the legend scale doesn't change each year - need to make decision around this bc min and max will not show most changes
#     height=600,
# )

# fig.update_layout(title="SF Business Openings Change from 2016 by Census Tract")
# fig.show()

In [155]:
#fig.write_html("sf_business_map.html")

In [156]:
#from google.colab import files
#files.download("sf_business_map.html")

In [157]:
# fig = px.choropleth_mapbox(
#     closed,
#     geojson=tracts_plot.set_index("GEOID").__geo_interface__,
#     locations="GEOID",
#     color="change_from_2016",
#     hover_name="GEOID",
#     animation_frame="year",
#     mapbox_style="carto-positron",
#     zoom=11,
#     center={"lat": 37.7749, "lon": -122.4194},
#     color_continuous_scale="Reds",
#     range_color=[-10,20],
#     height=600,
# )

# fig.update_layout(title="SF Business Closings Change from 2016 by Census Tract")
# fig.show()

In [158]:
tracts_plot

,GEOID,geometry,year,status,count,baseline,change_from_2016
0,06077003406,"POLYGON ((-121.309 38.02824, -121.30461 38.028...",0.0,0,0.0,0.0,0.0
1,06077004402,"POLYGON ((-121.27338 38.10811, -121.27286 38.1...",0.0,0,0.0,0.0,0.0
2,06077004600,"POLYGON ((-121.411 38.23193, -121.40976 38.232...",0.0,0,0.0,0.0,0.0
3,06077001400,"POLYGON ((-121.30988 37.98438, -121.30745 37.9...",0.0,0,0.0,0.0,0.0
4,06077003106,"POLYGON ((-121.37304 37.99888, -121.37139 38.0...",0.0,0,0.0,0.0,0.0
...,...,...,...,...,...,...,...
23068,06073015502,"POLYGON ((-116.85838 32.81723, -116.85471 32.8...",0.0,0,0.0,0.0,0.0
23069,06073008350,"POLYGON ((-117.20607 32.89062, -117.20435 32.8...",0.0,0,0.0,0.0,0.0
23070,06071010010,"POLYGON ((-117.36884 34.4413, -117.35738 34.45...",0.0,0,0.0,0.0,0.0
23071,06071001312,"POLYGON ((-117.61121 34.08488, -117.61112 34.0...",0.0,0,0.0,0.0,0.0


In [159]:
sf_tracts = gpd.clip(tracts_gdf, sf_poly)

In [163]:
business_tracts_gdf.to_parquet('/content/drive/MyDrive/C255_final_project/cleaned/rbl_GEOID_all_dates.parquet')
sf_tracts.to_parquet('/content/drive/MyDrive/C255_final_project/cleaned/sf_tracts_GEOID.parquet')

In [162]:
tracts_rde = business_tracts_gdf[business_tracts_gdf['naics_code'].str.contains('4400-4599|7100-7199|7220-7229', regex=True, na=False)]

tracts_rde.to_parquet('/content/drive/MyDrive/C255_final_project/cleaned/rbl_rde_GEOID_all_dates.parquet')